In [ ]:
!pip install openai

In [ ]:
import pandas as pd
from openai import OpenAI
import time
import os


# Konfigurasi
INPUT_PATH = r"C:\Users\Yohanes\Documents\Lab\IntentMining-python-3.11\dataset_anonim_phase7_2024.xlsx"
OUTPUT_PATH = r"C:\Users\Yohanes\Documents\Lab\IntentMining-python-3.11\dataset_anonim_phase7_2024_anonim_NER_openAI_2.xlsx"

# setx OPENAI_API_KEY "your_key"
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
client = OpenAI(api_key="")

# Prompt / PERINTAH
PERINTAH = (
    "Kamu adalah ahli NER (Named Entity Recognition) untuk teks bahasa Indonesia di domain statistik resmi. "
    "Kamu menguasai konsep SDMX dan berbagai kosakata dunia statistik, termasuk istilah teknis, nama lembaga, nama tempat, dan format penulisan angka serta tanggal yang umum digunakan dalam konteks statistik. "
    "Kamu juga memahami konteks pertanyaan yang sering muncul terkait data statistik, seperti pertanyaan tentang angka kemiskinan, jumlah penduduk, tingkat pengangguran, dan sebagainya. "
    "Tugasmu adalah menganalisis kalimat yang akan saya berikan dan melakukan anonimisasi kata per kata yang mungkin berelasi dengan kategori di bawah ini:\n"
    "PERSON: Nama orang (misal: Kepala BPS, Dr. Agus Santoso, kata ganti orang seperti: saya, kami, dll.)\n"
    "ORG: Organisasi / institusi resmi (misal: Badan Pusat Statistik, BPS Provinsi Jawa Barat, Kementerian Kesehatan, kata yang merujuk ke institusi/organisasi)\n"
    "REF_AREA: nama negara, tempat atau wilayah spesifik di Indonesia (misal: Jakarta Selatan, Kabupaten Sleman, Kecamatan Setiabudi, atau kata 'kabupaten', 'kota', 'kecamatan' yang tidak spesifik menyebut nama tempat)\n"
    "TIME_PERIOD: Tahun, Tanggal atau periode sensu/survei (misal: 15 Januari 2023, tahun 2024)\n"
    "NUMBER: Nomor referensi dokumen atau publikasi (misal: No. 123/BPS/2023)\n"
    "NORP: Nasionalitas, kelompok agama atau politik relevan data survei (misal: Warga Negara Indonesia, Islam, Partai Demokrat)\n"
    "FACTORY: Fasilitas atau gedung resmi (misal: Kantor BPS Pusat, Gedung Statistik Nasional)\n"
    "PRODUCT: Produk data, survei, sensus (misal: KBLI/kode KBLI, Susenas, Sakernas, SP2020, sensus penduduk, sensus ekonomi, data mikro, publikasi, buku pedoman)\n"
    "FILE_FORMAT: tipe file atau dokumen (misal: file xlsx/excel, dokumen pdf, file dbf, tabel data)\n"
    "ITEM: nama benda diluar [SUBJECT] atau objek yang terkait dengan data atau survei (misal: beras, minyak goreng, kendaraan, pakaian, mug, tumbler, boneka santa, jar, tea set, buku, nama-nama hewan,nama-nama item di kamus KBLI)\n"
    "INDICATOR: kata atau sekumpulan kata yang merepresentasi tema statistik (misal: Population,Migration,Labour,Education,Health,Income,consumption,Social,housing,Culture,Macroeconomic,System of National Accounts,Balance of payments,international investment position,finance,Monetary,Accounting,Business,Entrepreneurship,Sectoral statistics,Agriculture, forestry and fishing,Energy,Mining manufacturing,construction,Transport,Tourism,Banking, insurance,Wholesale and retail trade,Sectoral statistics,International trade,Prices)  \n"
    "LAW: Undang-undang atau regulasi terkait data (misal: UU Statistik 2010, Peraturan BPS No. 5/2021)\n"
    "LANGUAGE: Bahasa yang digunakan dalam publikasi (misal: Bahasa Indonesia, Bahasa Inggris)\n"
    "UNIT_MEASURE: satuan yang digunakan dalam data, misal: persen, jiwa, rupiah, responden, kilogram/kg, liter\n"
    "DATA_VALUE: Angka absolut atau statistik (misal: 5, 100, 1.000, 10.500) atau angka yang menyatakan nilai sebuah indikator atau pengukuran data statistik, misal 100, 140, 6000\n"
    "ATTRIBUTE: mengacu pada metadata dari data statistik, atau informasi tambahan pada data statistik, seperti konsep, definisi, metodologi \n"
    "FREQUENCY: Frekuensi atau interval waktu dalam survei (misal: harian, mingguan, bulanan, tahunan, triwulanan, kuartal, annual)\n"
    "OTHER_DIMENSION: Dimensi lain yang relevan dengan data atau survei (misal: jenis kelamin, kelompok umur, tingkat pendidikan, status pekerjaan)\n"
    "QUESTION_MODAL: Kata tanya misal: Berapa, Seberapa, Kapan, Di mana, Siapa, Bagaimana, termasuk 'apakah', 'bisakah', 'haruskah', 'mungkinkah', 'adakah', 'apakah mungkin'\n\n"
    "Identifikasi kata tanya dalam kalimat bahasa Indonesia dan tag dengan kode berikut: [WHAT], [WHO], [WHOM], [WHERE], [WHEN], [HOW], [WHY], [WHICH], [HOW_MANY], [HOW_MUCH], [WHOSE] "
    "Instruksi:\n"
    "- Analisis kalimat dan ganti kata spesifik dengan label entity dalam bentuk [LABEL]\n"
    "- Ganti kata tanya di kalimat dengan tag sesuai tabel. Pastikan perbedaan HOW dengan HOW_MANY dan HOW_MUCH sesuai konteks kalimat\n"
    "- Contoh: Berapa angka kemiskinan Indonesia 2025? output format: [(Berapa, HOW_MANY), (angka kemiskinan, INDICATOR), (Indonesia, REF_AREA), (2025, TIME_PERIOD)]: [HOW_MANY] [INDICATOR] [REF_AREA] [TIME_PERIOD]?\n"
    "- Output format harus: [(kata, label tag), (kata, label tag),..]: kalimat hasil anotasi [tabel tag] dan jangan tambahkan penjelasan apapun selain format contoh.\n"
    "- Jangan tambahkan penjelasan, tanda kutip, atau karakter lain. konsisten selalu pada output format\n"
    "Kalimat:\n"
)

# Load file XLSX
USERDATA = pd.read_excel(INPUT_PATH)

if "ner" not in USERDATA.columns:
    USERDATA["ner"] = ""

# Looping row
for idx, row in USERDATA.iterrows():
    SENTENCE = str(row["question"])
    
    try:
        response = client.responses.create(
            model="gpt-5",
            reasoning={"effort": "low"},
            instructions=PERINTAH,
            input=SENTENCE,
        )

        hasil_text = response.output_text.strip()

        USERDATA.at[idx, "ner"] = hasil_text

        print(f"[{idx+1}/{len(USERDATA)}] Done")

        time.sleep(0.1)

    except Exception as e:
        print(f"Error at row {idx}: {e}")
        USERDATA.at[idx, "ner"] = "ERROR"

# Save to XLSX
USERDATA.to_excel(OUTPUT_PATH, index=False)

print(f"Done! File tersimpan di: {OUTPUT_PATH}")